In [1]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.19.6
#   kernelspec:
#     display_name: Python (latest Jaspy)
#     language: python
#     name: jaspy
# ---

# AAA03: synthetic study, Student-t copula with a hump-shaped tau(w), weak signal

Design **nonmonotonic, weak**: Student-t copula (df = 4); hump-shaped tau(w) = 0 + 0.10 b(w) with b(w) = (exp(-w^2 / (2 * 0.75^2)) - e) / (1 - e) and e = exp(-1.5^2 / (2 * 0.75^2)), so tau = tau_hi at w = 0 and tau = tau_lo at |w| = 1.5, centred on the median of w. Only the SHAPE of tau(w) changes against the Student-t designs (same Student-t family): isolates the shape. A median split of w is blind to this structure by construction: both halves have the same average tau.

* **Sample size.** n = `N_TRAIN` training points per replication (`synthetic_study.N_TRAIN`, 2,000 by default, about the size of one training fold of the real data); `MC_REPS` independent data draws (Monte-Carlo replication), w ~ U(-1.5, 1.5); evaluation on one fixed fresh out-of-sample draw of 10,000 points. Metrics are means over the draws with the SD across draws.
* Models: seven parametric families (unconditional; Gaussian, t, Clayton, Gumbel, Gumbel (180°), Frank, Joe), binary split at the median of w (49 pairs by indexing; AIC-selected pair), Acar continuous with the same seven families, unconditional CODINE, two-net split CODINE, 3 C-CODINE and 6 C-CopFlow; one training seed per net and draw (the spread across draws includes the training noise). Hyper-parameters (weight decay, early stopping, batch size) are those of the real-data pipeline; nothing is re-tuned per design. The classical conditional-copula estimator of Gijbels, Veraverbeke and Omelka (2011) and the empirical checkerboard Bernstein copula (ECBC) of Lu and Ghosh (2024) are added as benchmarks. In the summary figures of AAA_summary the Acar model of the family that generated the data is shown as a reference (the best-fitting of the seven Acar models in the two mixture designs).
* Metrics: oracle CVPE (true conditional expectation) and each model's excess over it; tau(w) RMSE; finite-threshold tail RMSE (chi(q = 0.95) per corner against the true copula); Copula Energy Score.
* Misspecification here means a fitted family that does not contain the DGP, or the wrong conditioning structure (a median split of a hump). Noisy inputs are not covered (future work).
* `THESIS_SYNTH_REPS` changes the number of data draws (results are cached per setting). `THESIS_SMOKE=1` is the plumbing test (tiny everything, never report from it).

## Imports and settings

In [2]:
import os
os.environ["OMP_NUM_THREADS"] = "1"; os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"; os.environ["NUMEXPR_NUM_THREADS"] = "1"
import warnings; warnings.filterwarnings('ignore')
import pandas as pd

import fold_pipeline as FP
import gate_config as GC
import synthetic_study as S
import synthetic_report as R
import gijbels_copula as GIJ_M
import ecbc_copula as ECBC_M
from run_support import save_artifact, SMOKE, MAX_WORKERS, resolve_max_workers   # override with THESIS_SMOKE / THESIS_MAX_WORKERS

NOTEBOOK = "AAA03_synthetic_nonmonotonic_weak"
DGP = "3c"                       # internal key of the design in synthetic_study.DGPS
TAG = "nonmonotonic_weak"                       # name stem of artifacts, tables and figures

MC_REPS = int(os.environ.get("THESIS_SYNTH_REPS", 3 if SMOKE else S.MC_REPS))     # independent data draws
N_TRAIN = 300 if SMOKE else S.N_TRAIN                                            # training points per draw (2,000 by default)
N_OOS = 300 if SMOKE else S.N_OOS
BATCH = FP.BATCH_DIR / "synthetic"                                                 # resume files of all synthetic notebooks
spec = S.DGPS[DGP]
DESC = f"{spec['family'].replace('_', '-')} copula, {spec['shape']} tau(w), {spec['strength']} signal"
SETTING = f"n = {N_TRAIN:,} training points, {MC_REPS} independent data draws, {N_OOS:,} out-of-sample points"
print(f"{TAG}: {DESC} | {SETTING}" + (" | SMOKE (plumbing test)" if SMOKE else ""))

nonmonotonic_weak: t copula, hump tau(w), weak signal | n = 2,000 training points, 10 independent data draws, 10,000 out-of-sample points


## Data and the check of tau(w) before any model is fitted

In [3]:
BUNDLES = S.make_bundles(DGP, reps=MC_REPS, n_train=N_TRAIN, n_oos=N_OOS)
bundle = BUNDLES[0]                     # first draw: used for the checks and the figures
chk = S.check_tau_in_bins(bundle['u1'], bundle['u2'], bundle['w'].astype(float), spec)
dev = float((chk.tau_empirical - chk.tau_formula).abs().max())
zmax = float(((chk.tau_empirical - chk.tau_formula).abs() / chk.tau_se).max())
print(f"sampler check: max |empirical - formula| of tau over w bins = {dev:.3f} (largest z-score {zmax:.1f}; flagged above 4)",
      "OK" if zmax < 4 else "-- CHECK THE SAMPLER BEFORE FITTING")
print("oracle CVPE (true conditional expectation, out-of-sample):", round(float(bundle['oracle_loss'].mean()), 5))
tau_ref = float(S.tau_true(1.5, spec)) if spec['shape'] == 'monotone' else float(S.tau_true(0.0, spec))   # w = 1.5 (monotone) / the peak w = 0 (hump)
lam, corner = S.lambda_true(tau_ref, spec)
print(f"asymptotic tail coefficient of the DGP at tau = {tau_ref:.3f}: {lam:.3f} (corner {corner})")

sampler check: max |empirical - formula| of tau over w bins = 0.111 (largest z-score 2.5; flagged above 4) OK
oracle CVPE (true conditional expectation, out-of-sample): 0.16439
asymptotic tail coefficient of the DGP at tau = 0.100: 0.114 (corner UU and LL)


## Parametric and Acar models (deterministic fits; one job per data draw)

In [4]:
FP.BATCH_DIR.mkdir(parents=True, exist_ok=True)
with R.quiet():
    res = FP.run_batch([S.SynthJob(bundle=b, kind='param_acar', smoke=SMOKE, n_eval=8 if SMOKE else GC.ACAR_N_EVAL) for b in BUNDLES], max_workers=MAX_WORKERS,
                       batch_dir=BATCH, fn=S.parametric_acar_job,
                       key_fn=lambda j: S.param_key(j['bundle']), desc='parametric + Acar')
PARAM_OUTS = {b['rep']: res[S.param_key(b)] for b in BUNDLES}
PARAM_OUT = PARAM_OUTS[0]
_u = pd.Series([PARAM_OUTS[r]['selected_uncond'] for r in sorted(PARAM_OUTS)]).value_counts().to_dict()
_p = pd.Series(["/".join(PARAM_OUTS[r]['selected_pair']) for r in sorted(PARAM_OUTS)]).value_counts().to_dict()
print(f"parametric + Acar done for {len(PARAM_OUTS)} draws (draw 0: {PARAM_OUT['runtime']:.0f}s)")
print("AIC-selected unconditional family (draws):", _u, "| AIC-selected low/high pair (draws):", _p)

parametric + Acar done for 10 draws (draw 0: 445s)
AIC-selected unconditional family (draws): {'t': 10} | AIC-selected low/high pair (draws): {'t/t': 10}


## Neural networks: 12 nets per data draw

In [5]:
WD = S.DEFAULT_WD            # one calibrated value per group (notebook 03), identical to notebook 04
jobs = S.build_neural_synth_jobs(BUNDLES, WD, seeds=S.TRAIN_SEEDS, smoke=SMOKE)
with R.quiet():
    NEURAL_OUT = FP.run_batch(jobs, max_workers=MAX_WORKERS, batch_dir=BATCH, fn=S.neural_synth_job,
                              key_fn=S.synth_key, desc=f'{TAG} neural')
print(f"{len(NEURAL_OUT)} of {len(jobs)} network trainings available (weight decay: C-CODINE {FP.WD_CCODINE:g}, C-CopFlow affine {FP.WD_CCOPFLOW_AFFINE:g}, C-CopFlow spline {FP.WD_CCOPFLOW_SPLINE:g})")

120 of 120 network trainings available (weight decay: C-CODINE 0.001, C-CopFlow affine 0.01, C-CopFlow spline 0.01)


## Copula Energy Score of the networks (additional metric, reads the stored networks)

Lower is better; the raw joint score of the PIT pair under the fitted distribution is computed by deterministic quadrature (no Monte Carlo noise); the margin-calibrated score is stored next to it. Networks whose weights were not stored are listed as *not reconstructable* and get no CES; they are not retrained here.

In [6]:
with R.quiet():
    CES_RES = S.run_ces_synth(jobs, max_workers=MAX_WORKERS, K=32 if SMOKE else 64, G=21 if SMOKE else 81, batch_dir=BATCH)
_st = pd.Series([r['status'] for r in CES_RES.values()]).value_counts().to_dict()
print("Copula Energy Score of the stored networks:", _st, "" if set(_st) <= {'ok'} else "(networks that are not reconstructable get no CES; rerun their training once)")

Copula Energy Score of the stored networks: {'ok': 120} 


## Summary over the data draws

In [7]:
SUM, SUM_LONG = S.summarize_mc(BUNDLES, PARAM_OUTS, NEURAL_OUT, ces_results=CES_RES)
print(f"oracle CVPE {SUM.attrs['oracle_cvpe']:.5f}; oracle CES {SUM.attrs.get('oracle_ces', float('nan')):.5f}; independence CES {SUM.attrs.get('independence_ces', float('nan')):.5f}")

oracle CVPE 0.16439; oracle CES 0.25911; independence CES 0.25928


## Classical nonparametric benchmark: Gijbels, Veraverbeke and Omelka (2011) transformed estimator

Specification of the paper: **local-linear weights**, triweight kernel, conditional margins with bandwidths g1, g2 and the copula bandwidth h from the **AMISE plug-in rule** of their Section 2.3 (iterated pilot bandwidths), built from the asymptotic bias and variance of Veraverbeke, Omelka and Gijbels (2011). The authors' own bandwidth program is unpublished, so the rule is re-derived from the published asymptotics (g does not use `lokern`). The paper states that local-linear weights need not be positive (Veraverbeke et al. 2011, p. 768) and gives no rule for this; conditional cdf values are clipped to [0, 1] and negative cell masses are clipped and renormalised. Bandwidths are capped at the covariate range. The out-of-sample draw is used only for scoring.

In [8]:
with R.quiet():
    GIJ, GIJ_FAIL = GIJ_M.run_gijbels(BUNDLES, FP.BATCH_DIR / 'gijbels', max_workers=resolve_max_workers(n_tasks=len(BUNDLES)), K=32 if SMOKE else 64, G=21 if SMOKE else 81)
print(f"Gijbels runs ok: {len(GIJ)}" + (f" | FAILED: {GIJ_FAIL}" if GIJ_FAIL else ""))

Gijbels runs ok: 10


## Nonparametric benchmark: empirical checkerboard Bernstein copula (ECBC) of Lu and Ghosh (2024)

The conditional copula is estimated as the derivative with respect to the covariate of a three-dimensional empirical checkerboard Bernstein copula of the covariate-adjusted pseudo-observations and the covariate rank (Lu and Ghosh, 2024, eqs. 12-16). Its conditional margins are made exactly uniform by the Sklar inversion of eq. 16, so it is a genuine copula at every w. The estimator is averaged over five draws of the degrees (l1, l2, m) from the hierarchical shifted-Poisson prior of the paper, with stratified exponents alpha in (1/3, 2/3) and a fixed seed (see the module docstring of `ecbc_copula`). Scored on the same K x K cell grid and with the same oracle as the Gijbels benchmark, so the two excess columns are comparable with each other but not with the exact CVPE of the main table. The out-of-sample draw is used only for scoring.

In [9]:
with R.quiet():
    ECBC, ECBC_FAIL = ECBC_M.run_ecbc(BUNDLES, FP.BATCH_DIR / 'ecbc', max_workers=resolve_max_workers(n_tasks=len(BUNDLES)), K=32 if SMOKE else 64, G=21 if SMOKE else 81)
print(f"ECBC runs ok: {len(ECBC)}" + (f" | FAILED: {ECBC_FAIL}" if ECBC_FAIL else ""))

ECBC runs ok: 10


## Fitted Kendall's tau(w): peak locations

In [10]:
H = S.hump_diagnostics(bundle, PARAM_OUT, {k: v for k, v in NEURAL_OUT.items() if v['rep'] == 0})   # first draw
print("peak location of each curve:", {k: round(v, 2) for k, v in H['peak_location'].items()})

peak location of each curve: {'truth': 0.0, 'binary (AIC-selected pair)': 0.0, 'unconditional (AIC-selected)': -1.4, 'Acar (continuous): gaussian': 0.2, 'Acar (continuous): student': 0.4, 'Acar (continuous): clayton': 0.2, 'Acar (continuous): gumbel': 0.4, 'Acar (continuous): gumbel180': 0.2, 'Acar (continuous): frank': 0.4, 'Acar (continuous): joe': 0.6, 'C-CopFlow (w only, affine)': -1.4, 'C-CopFlow (FiLM, affine)': 1.4, 'C-CopFlow (spline basis, affine)': -0.4, 'C-CopFlow (FiLM, spline coupling)': 1.4, 'C-CopFlow (w only, spline coupling)': 1.4, 'C-CopFlow (spline basis, spline coupling)': -0.4, 'C-CODINE (FiLM)': -1.2, 'C-CODINE (spline basis)': 0.4, 'C-CODINE (w only)': 0.4}


### The binary split is blind to the hump

Both halves of a symmetric hump have the same average tau, so the median split's two-value curve is flat in expectation. The numbers:

In [11]:
w_half = bundle['w'].astype(float) < bundle['w_median']
print("true mean tau, low half :", round(float(bundle['tau'][w_half].mean()), 3))
print("true mean tau, high half:", round(float(bundle['tau'][~w_half].mean()), 3))
sel = PARAM_OUT['selected_pair']
c = PARAM_OUT['models'][f"Divergence split: {sel[0]} (low) / {sel[1]} (high)"]['tau_curve']
print("AIC-selected split, fitted tau low / high:", round(float(c[0]), 3), round(float(c[-1]), 3))

true mean tau, low half : 0.054
true mean tau, high half: 0.052
AIC-selected split, fitted tau low / high: 0.054 0.067


## Save

In [12]:
save_artifact(f"synthetic_{TAG}", summary=SUM, summary_by_draw=SUM_LONG, param=PARAM_OUT,
              neural={k: {kk: vv for kk, vv in r.items()} for k, r in NEURAL_OUT.items()},
              spec=spec, curves=H, w_eval=S.W_EVAL, oracle_cvpe=SUM.attrs['oracle_cvpe'], tau_bins=chk,
              gijbels=GIJ if len(GIJ) else None, ecbc=ECBC if len(ECBC) else None,
              seeds=list(S.TRAIN_SEEDS), mc_reps=MC_REPS, n_train=N_TRAIN, dgp=DGP, smoke=SMOKE)
print("saved synthetic_" + TAG)

saved synthetic_nonmonotonic_weak
